In [ ]:
from astropy.io import fits
import numpy as np
import emcee
import corner
import math
from astropy.coordinates import Angle
import astropy.units as units
import scipy.special as sp
import warnings
import itertools
from uncertainties import ufloat
from uv_fit import *
import sigfig

In [ ]:
# def limiting_value(param_name, tuple, param_chain, n_walkers, param_priors):
#     median = tuple[0]
#     sd = tuple[1]
#     last_n = param_chain[-n_walkers:]

#     # adjusting for hardcoded priors
#     if param_priors is None:
#         if param_name in ['vis_sigma', 'vis_r', 'ratio']:
#             param_priors = [0, None]
#         if param_name == 'vis_theta':
#             param_priors = [-np.pi/2, np.pi/2]
#     if param_priors is not None:
#         if param_priors[0] is None:
#             if param_name in ['vis_sigma', 'vis_r', 'ratio']:
#                 param_priors = [0, param_priors[1]]
#             if param_name == 'vis_theta':
#                 param_priors = [-np.pi/2, param_priors[1]]
#         if param_priors[1] is None:
#             if param_name == 'vis_theta':
#                 param_priors = [param_priors[0], np.pi/2]
#             if param_name == 'ratio':
#                 param_priors = [param_priors[0], 1]
#         if param_priors[0] is not None:
#             if param_name in ['vis_sigma', 'vis_r', 'ratio']:
#                 param_priors = [max(param_priors[0], 0), param_priors[1]]
#             if param_name == 'vis_theta':
#                 param_priors = [max(param_priors[0], -np.pi/2), np.pi/2]
#         if param_priors[1] is not None:
#             if param_name == 'ratio':
#                 param_priors = [param_priors[0], min(param_priors[1], 1)]
#             if param_name == 'vis_theta':
#                 param_priors = [param_priors[0], min(param_priors[1], np.pi/2)]

#     if all(last_n > median): # running away to higher values
#         return ('upper', np.percentile(param_chain, 99.7))
#     elif all(last_n < median):  # running away to lower values
#         return ('lower', np.percentile(param_chain, 0.3))

#     if param_priors is not None:
#         if param_priors[0] is not None and all(last_n < param_priors[0] + sd/2): # hugging lower prior
#             return ('lower prior', np.percentile(param_chain, 0.3))
#         elif param_priors[1] is not None and all(last_n > param_priors[1] - sd/2): # hugging upper prior
#             return ('upper prior', np.percentile(param_chain, 99.7))

#     if param_name == 'vis_theta' and param_priors == [-np.pi/2, np.pi/2]: # special case for angle wrapping
#         neg_vis_thetas = [theta for theta in param_chain if theta < 0]
#         pos_vis_thetas = [theta for theta in param_chain if theta >= 0]
#         neg_med = np.median(neg_vis_thetas) if neg_vis_thetas else None
#         pos_med = np.median(pos_vis_thetas) if pos_vis_thetas else None
#         if neg_med is not None and pos_med is not None:
#             if abs(abs(neg_med) - pos_med) < 10 * np.pi/180:
#                 return('angle wrap', -np.pi/2)
#     return None

In [ ]:
P_PARAMS = ['peak', 'ra', 'dec']
C_PARAMS = ['peak', 'ra', 'dec', 'sigma']
G_PARAMS = ['peak', 'ra', 'dec', 'sigma', 'ratio', 'vis_theta']
D_PARAMS = ['peak', 'ra', 'dec', 'r']

SOURCE_TYPES = {'p': [3, p_p0, p_prior, p_model, P_PARAMS], \
                'c': [4, c_p0, c_prior, c_model, C_PARAMS], \
                'g': [6, g_p0, g_prior, g_model, G_PARAMS], \
                'd': [4, d_p0, d_prior, d_model, D_PARAMS]}

In [ ]:
def sim_uv_fit(info, vis, sources: list, priors: list = None, clean_output=True, corner_plot=True, additional_runs: int = 2):
    # priors = [[(i0_min, i0_max), (l0_min, l0_max), (m0_min, m0_max), (width_param_min, width_param_max), (ratio_min, ratio_max), (theta_min, theta_max)], ...]
    # but (tuple) for exclusive and [list] for inclusive
    # TODO: documentation
    '''
    Fit UV data from a FITS file with specified source types using MCMC.

    Parameters
    ----------
    fits_file (str): Path to the UV FITS file.
    sources (list): List of source types to fit. Each source type should be one of
                    'p' (point), 'c' (circular gaussian), 'g' (gaussian), 'd' (disk), or 'any' (try all and pick best fit).
    '''
    # Check additional_runs
    if additional_runs < 0:
        raise ValueError("additional_runs must be a non-negative integer.")

    # Check priors format
    if priors is not None:
        if len(priors) != len(sources):
            raise ValueError("Length of priors must match length of sources.")
        for i in range(len(priors)):
            if priors[i] is not None:
                if type(priors[i]) is not list:
                    raise ValueError("Each element in priors must be None or a list corresponding to a source.")
                if len(priors[i]) != 6:
                    raise ValueError("Each prior list must have 6 elements corresponding to ranges for peak, RA, declination, width parameter, ratio, and angle.")
                for i in range(len(priors[i])):
                    if priors[i][i] is not None:
                        if type(priors[i][i]) not in [list, tuple]:
                            raise ValueError("Each prior range must be None, a list (inclusive), or a tuple (exclusive).")
                        if len(priors[i][i]) != 2:
                            raise ValueError("Each prior range list or tuple must have exactly two elements: min and max.")

    # Check input source types
    if len(sources) == 0:
        raise ValueError("No sources specified. Try specifying one or more sources of type \
                         'p' (point), 'c' (circular gaussian), 'g' (gaussian), 'd' (disk), or 'any' (try all and pick best fit).")
    for source in sources:
        if source not in SOURCE_TYPES and source != 'any':
            raise ValueError(f"Source type '{source}' is not recognized. Source type must be one of the following: \
                            'p' (point), 'c' (circular gaussian), 'g' (gaussian), 'd' (disk), or 'any' (try all and pick best fit).")

    vis_priors = []
    if priors is None:
        priors = [None] * len(sources)
    for i in range(len(priors)):
        mini_vis_priors = []
        if priors[i] is not None:
            for j in range(len(priors[i])):
                if priors[i][j] is not None:
                    if j == 0:  # peak, keep as is
                        mini_vis_priors.append(priors[i][j])
                    elif j in [1, 2, 3]:  # ra, dec, width parameter
                        # convert from arcsec to radian
                        rad_min = float(Angle(priors[i][j][0], units.arcsec).to(units.radian).value)
                        rad_max = float(Angle(priors[i][j][1], units.arcsec).to(units.radian).value)
                        if type(priors[i][j]) is tuple:
                            mini_vis_priors.append((rad_min, rad_max))
                        else:
                            mini_vis_priors.append([rad_min, rad_max])
                    elif j == 4:  # ratio
                        mini_vis_priors.append(priors[i][j])
                    elif j == 5:  # angle
                        mini_vis_priors.append(priors[i][j] * np.pi/180) # convert from degrees to radians
                else:
                    mini_vis_priors.append(None)
            vis_priors.append(mini_vis_priors)
        else: # nothing to convert
            vis_priors.append([[None, None]] * 6)

    # Extract data from fits file
    cdelt1 = info['CDELT1']
    cunit1 = info['CUNIT1']
    naxis1 = info['NAXIS1']

    bmaj = info['BMAJ'] # cunit1
    bmin = info['BMIN'] # cunit1
    rad_bmaj = Angle(bmaj, cunit1).to(units.radian).value
    rad_bmin = Angle(bmin, cunit1).to(units.radian).value
    rad_barea = np.pi * rad_bmaj * rad_bmin / (4 * np.log(2))
    rad_pix = float(Angle(cdelt1, cunit1).to(units.radian).value)
    int_peaks = info['int_peak_val']
    int_coords = info['int_peak_coord']
    ext_peaks = info['ext_peak_val']
    ext_coords = info['ext_peak_coord']

    int_info = list(zip(int_peaks, int_coords))
    if type(ext_peaks) is list:
        ext_info = list(zip(ext_peaks, ext_coords))
    else:
        ext_info = []
    all_peaks = int_info + ext_info # list of tuples (peak_value, (l_coord, m_coord))
    all_peaks.sort(reverse=True) # sort by peak value
    n_peaks = len(all_peaks)
    if n_peaks < len(sources):
        warnings.warn(f"Number of detected peaks ({n_peaks}) is less than number of sources to fit ({len(sources)}).")

    freq_bin, u, v, re, im, w = [], [], [], [], [], []
    for row in vis:
        freq_bin_data, u_data, v_data, re_data, im_data, w_data = row
        freq_bin.append(int(freq_bin_data))
        u.append(int(u_data))
        v.append(int(v_data))
        re.append(float(re_data/w_data))
        im.append(float(im_data/w_data))
        w.append(float(w_data))

    # Adding in conjugate half of data
    freq_bin *= 2
    neg_u = [-1 * val for val in u]
    u += neg_u
    neg_v = [-1 * val for val in v]
    v += neg_v
    re *= 2
    neg_im = [-1 * val for val in im]
    im += neg_im
    w *= 2

    freq_bin = np.array(freq_bin)
    u = np.array(u)
    v = np.array(v)
    re = np.array(re)
    im = np.array(im)
    w = np.array(w)

        # Estimate total flux from small baselines
    small_baselines = []
    q = np.sqrt(u**2 + v**2)
    baseline_indices = np.argsort(q, axis=None)
    small_baselines_indices = baseline_indices[:len(baseline_indices)//20]  # smallest 5% of baselines
    for i in small_baselines_indices:
        small_baselines.append(np.sqrt(im[i]**2 + re[i]**2))
    total_flux_median = np.median(small_baselines)
    total_flux_mean = np.mean(small_baselines)
    total_flux_sd = np.std(small_baselines)
    if abs(total_flux_median - total_flux_mean) > total_flux_sd:
        total_flux = total_flux_median
    else:
        total_flux = total_flux_mean

    # All possible permutations
    n_sources = len(sources)
    sample_space = list(SOURCE_TYPES.keys()) * n_sources
    all_permutations = list(itertools.permutations(sample_space, n_sources))

    for i in range(n_sources):
        if sources[i] != 'any':
            all_permutations = [p for p in all_permutations if p[i] == sources[i]] # remove unwanted permutations
    all_permutations = list(set(all_permutations)) # remove duplicates

    all_results = []
    for permutation in all_permutations:
        # Calculate n_params and n_walkers
        n_params = 0
        for i in range(n_sources):
            source = permutation[i]
            n_params += SOURCE_TYPES[source][0]
        n_walkers = 2 * n_params

        # Initial guesses
        for i in range(n_sources):
            source = permutation[i]
            peak = all_peaks[i][0] if i < n_peaks else all_peaks[-1][0]
            coord0 = all_peaks[i][1] if i < n_peaks else all_peaks[-1][1]
            rad_coord = (float(Angle(coord0[0], units.arcsec).to(units.radian).value), float(Angle(coord0[1], units.arcsec).to(units.radian).value))
            if i == 0:
                p0 = SOURCE_TYPES[source][1](peak, rad_coord, rad_pix, rad_barea, total_flux, n_walkers)
            else:
                mini_p0 = SOURCE_TYPES[source][1](peak, rad_coord, rad_pix, rad_barea, total_flux, n_walkers)
                if i >= n_peaks: # edit ra, dec initial guesses
                    for j in range(n_walkers):
                        mini_p0[j,1] = np.random.uniform(-naxis1/2*rad_pix, naxis1/2*rad_pix)
                        mini_p0[j,2] = np.random.uniform(-naxis1/2*rad_pix, naxis1/2*rad_pix)
                p0 = np.append(p0, mini_p0, axis=1)

        # Set up and run MCMC
        n_steps = 100
        sampler = emcee.EnsembleSampler(n_walkers, n_params, log_probability, args=(permutation, vis_priors, re, im, u, v, w, rad_bmaj, rad_bmin))
        try:
            state = sampler.run_mcmc(p0, n_steps)
        except emcee.autocorr.AutocorrError:
            pass
        tau = sampler.get_autocorr_time(quiet=True)
        if np.isnan(tau).all():
            warnings.warn(f"Autocorrelation time for first run of {permutation} could not be estimated; all values are NaN.", RuntimeWarning)
            all_results.append({'permutation': permutation, 'n_params': n_params, 'result': None, 'chi2': np.inf, 'chain': None})
            continue
        int_tau = math.ceil(np.nanmax(tau))
        steps_to_50_tau = abs(int_tau * 50 - n_steps)
        sampler.run_mcmc(state, steps_to_50_tau)
        chain = sampler.get_chain(discard = int_tau * 10, flat=True)
        log_probs = sampler.get_log_prob(discard = int_tau * 10, flat=True)
        max_prob_index = np.argmax(log_probs)

        # Find parameter estimates and uncertainties and calculate chi2
        result = {}
        model = 0.0
        start = 0
        for i in range(n_sources):
            source = permutation[i]
            n_source_params = SOURCE_TYPES[source][0]
            source_chain = chain[:, start:start+n_source_params]
            source_result = {'type': source}
            temp_medians = [] # to store medians
            temp_bests = {} # to store best values (that maximimize probability)
            temp_max_probs = [] # to store max prob values
            for j in range(n_source_params):
                samples = source_chain[:, j]
                temp_max_probs.append(samples[max_prob_index])
                samples_med = np.median(samples)
                samples_sd = np.nanstd(samples)
                param_name = SOURCE_TYPES[source][4][j]
                source_result[param_name] = (float(samples_med), float(samples_sd))
                temp_bests[param_name] = float(samples[max_prob_index])
                temp_medians.append(samples_med)
            source_result['best'] = temp_bests
            model += SOURCE_TYPES[source][3](temp_max_probs, u, v, rad_bmaj, rad_barea)
            result[f'source_{i+1}'] = source_result
            start += n_source_params
        chi2 = float(np.sum(w * ((re - model.real)**2 + (im - model.imag)**2)))

        all_results.append({'permutation': permutation, 'n_params': n_params, 'result': result, 'chi2': chi2, 'chain': chain})

    # Rank permutations by chi2
    all_results.sort(key=lambda x: x['chi2']) # lowest to highest chi2
    best_perm = all_results[0]['permutation']
    best_result = all_results[0]['result']

    # Do it again with refined initial guesses, if requested
    for reps in range(additional_runs):  # two additional refinement iterations
        second_results = []
        for permutation_info in all_results:
            # Calculate n_params and n_walkers
            permutation = permutation_info['permutation']
            chain = permutation_info['chain']
            n_params = 0
            for i in range(n_sources):
                source = permutation[i]
                n_params += SOURCE_TYPES[source][0]
            n_walkers = 2 * n_params

            # New initial guesses from previous results
            for i in range(n_sources):
                source = permutation[i]
                coord0 = all_peaks[i][1] if i < n_peaks else all_peaks[-1][1]
                rad_coord = (float(Angle(coord0[0], units.arcsec).to(units.radian).value), float(Angle(coord0[1], units.arcsec).to(units.radian).value))
                if permutation_info['result'] is None: # fitting didn't happen, so can't actually refine
                    peak = all_peaks[i][0] if i < n_peaks else all_peaks[-1][0]
                    if i == 0:
                        p1 = SOURCE_TYPES[source][1](peak, rad_coord, rad_pix, rad_barea, total_flux, n_walkers)
                    else:
                        mini_p1 = SOURCE_TYPES[source][1](peak, rad_coord, rad_pix, rad_barea, total_flux, n_walkers)
                        if i >= n_peaks: # edit ra, dec initial guesses
                            for j in range(n_walkers):
                                mini_p1[j,1] = np.random.uniform(-naxis1/2*rad_pix, naxis1/2*rad_pix)
                                mini_p1[j,2] = np.random.uniform(-naxis1/2*rad_pix, naxis1/2*rad_pix)
                        p1 = np.append(p1, mini_p1, axis=1)
                else:
                    rad_position = best_result[f'source_{i+1}']['best']['ra'], best_result[f'source_{i+1}']['best']['dec']
                    source_result = permutation_info['result'][f'source_{i+1}']
                    best_params = []
                    med_sd = []
                    point_intensity = None
                    for param_name in SOURCE_TYPES[source][4]:
                        med_sd.append(source_result[param_name])
                        best_params.append(source_result[param_name][0])

                    # use p fitting to help c/g/d fitting if p chi2 was better than c/g/d chi2 in first run
                    resolved = True
                    if source != 'p':
                        if best_params[3] < rad_bmaj/2:  # conditions for unresolved source
                            print(f"Source {source} is unresolved")
                            resolved = False
                    if not resolved:
                        temp_perm = best_perm[:i] + ('p',) + best_perm[i+1:]
                        if temp_perm == best_perm:
                            point_intensity = best_result[f'source_{i+1}']['best']['peak']
                        else:
                            point_intensity = sim_uv_fit(info, vis, list(temp_perm), priors=priors, clean_output=True, corner_plot=False, additional_runs=0)[0]['result'][f'source_{i+1}']['peak'][0]

                    # use c fitting to help g fitting if c chi2 was better than g chi2 in first run
                    c_peak = None
                    c_sigma = None
                    if source == 'g' and best_perm[i] == 'c':
                        c_peak = best_result[f'source_{i+1}']['best']['peak']
                        c_sigma = best_result[f'source_{i+1}']['best']['sigma']
                    if i == 0:
                        p1 = all_p1(med_sd, resolved, point_intensity, c_peak, c_sigma, rad_position, rad_bmaj, rad_pix, n_walkers, chain)
                    else:
                        p1 = np.append(p1, all_p1(med_sd, resolved, point_intensity, c_peak, c_sigma, rad_position, rad_bmaj, rad_pix, n_walkers, chain), axis=1)

                    # edit vis_priors if unresolved source
                    if not resolved:
                        if vis_priors[i] is None:
                            vis_priors[i] = [(None, None)] * 6
                        vis_priors[i][1] = (-rad_pix+rad_coord[0], rad_pix+rad_coord[0]) # ra within one pixel of image domain result
                        vis_priors[i][2] = (-rad_pix+rad_coord[1], rad_pix+rad_coord[1]) # dec within one pixel of image domain result

            # Set up and run MCMC
            n_steps = 100
            sampler1 = emcee.EnsembleSampler(n_walkers, n_params, log_probability, args=(permutation, vis_priors, re, im, u, v, w, rad_bmaj, rad_bmin))
            try:
                state = sampler1.run_mcmc(p1, n_steps)
            except emcee.autocorr.AutocorrError:
                pass
            tau = sampler1.get_autocorr_time(quiet=True)
            if np.isnan(tau).all():
                warnings.warn(f"Autocorrelation time for second run of {permutation} could not be estimated; all values are NaN.", RuntimeWarning)
                second_results.append({'permutation': permutation, 'n_params': n_params, 'result': None, 'chi2': np.inf, 'chain': None})
                continue
            int_tau = math.ceil(np.nanmax(tau))
            steps_to_50_tau = abs(int_tau * 50 - n_steps)
            sampler1.run_mcmc(state, steps_to_50_tau)
            chain1 = sampler1.get_chain(discard = int_tau * 10, flat=True)
            log_probs1 = sampler1.get_log_prob(discard = int_tau * 10, flat=True)
            max_prob_index1 = np.argmax(log_probs1)

            # Find parameter estimates and uncertainties and calculate chi2
            result = {}
            model = 0.0
            start = 0
            for i in range(n_sources):
                source = permutation[i]
                n_source_params = SOURCE_TYPES[source][0]
                source_chain = chain1[:, start:start+n_source_params]
                source_result = {'type': source}
                temp_medians = [] # to store medians for chi2 calculation
                temp_bests = {} # to store best values (that maximimize probability)
                temp_max_probs = [] # to store max prob values
                for j in range(n_source_params):
                    samples = source_chain[:, j]
                    temp_max_probs.append(samples[max_prob_index1])
                    samples_med = np.median(samples)
                    samples_sd = np.nanstd(samples)
                    param_name = SOURCE_TYPES[source][4][j]
                    source_result[param_name] = (float(samples_med), float(samples_sd))
                    temp_bests[param_name] = float(samples[max_prob_index1])
                    temp_medians.append(samples_med)
                source_result['best'] = temp_bests
                model += SOURCE_TYPES[source][3](temp_max_probs, u, v, rad_bmaj, rad_barea)
                result[f'source_{i+1}'] = source_result
                start += n_source_params
            chi2 = float(np.sum(w * ((re - model.real)**2 + (im - model.imag)**2)))
            second_results.append({'permutation': permutation, 'n_params': n_params, 'result': result, 'chi2': chi2, 'chain': chain1})
        all_results = []
        for permutation_info in second_results:
            all_results.append(permutation_info)
        # Rank permutations by chi2
        all_results.sort(key=lambda x: x['chi2']) # lowest to highest chi2
        best_perm = all_results[0]['permutation']
        best_result = all_results[0]['result']

    # Bayesian Information Criterion
    n = len(re)
    for permutation_info in all_results:
        k = permutation_info['n_params']
        chi2 = permutation_info['chi2']
        if chi2 is np.inf:
            permutation_info['bic'] = np.inf
            continue
        bic = k * np.log(n) + chi2
        permutation_info['bic'] = float(bic)
    all_results.sort(key=lambda x: x['bic']) # lowest to highest BIC

    if clean_output:
        for permutation_info in all_results:
            result = permutation_info['result']
            start = 0
            permutation_chain = permutation_info['chain']
            if result is None:
                continue
            for i in range(n_sources):
                source_key = f'source_{i+1}'
                source_result = result[source_key]
                source_type = source_result['type']
                source_params = SOURCE_TYPES[source_type][4]
                n_source_params = SOURCE_TYPES[source_type][0]
                n_walkers = 2 * n_source_params
                source_chain = permutation_chain[:, start:start+n_source_params]
                source_priors = vis_priors[i]

                # peak
                peak_chain = source_chain[:, 0]
                peak_prior = source_priors[0]
                peak_limit = limiting_value('peak', source_result['peak'], peak_chain, n_walkers, peak_prior)
                source_result['peak'] = round_tuple((source_result['peak'][0], source_result['peak'][1]))
                if peak_limit is not None:
                    source_result['peak'] = (source_result['peak'][0], source_result['peak'][1], \
                                            (peak_limit[0], sigfig.round(peak_limit[1], sigfigs=3)))

                # convert ra, dec to arcsec
                ra_chain = source_chain[:, 1]
                dec_chain = source_chain[:, 2]
                ra_prior = source_priors[1]
                dec_prior = source_priors[2]
                ra_limit = limiting_value('ra', source_result['ra'], ra_chain, n_walkers, ra_prior)
                dec_limit = limiting_value('dec', source_result['dec'], dec_chain, n_walkers, dec_prior)
                source_result['ra'] = round_tuple(tuple([float(Angle(l, units.radian).to(units.arcsec).value) for l in source_result['ra']]))
                if ra_limit is not None:
                    source_result['ra'] = (source_result['ra'][0], source_result['ra'][1], \
                                           (ra_limit[0], sigfig.round(float(Angle(ra_limit[1], units.radian).to(units.arcsec).value), sigfigs=3)))
                source_result['dec'] = round_tuple(tuple([float(Angle(m, units.radian).to(units.arcsec).value) for m in source_result['dec']]))
                if dec_limit is not None:
                    source_result['dec'] = (source_result['dec'][0], source_result['dec'][1], \
                                            (dec_limit[0], sigfig.round(float(Angle(dec_limit[1], units.radian).to(units.arcsec).value), sigfigs=3)))

                if source_type != 'p': # convert visibility width to image width in arcsec
                    width_chain = source_chain[:, 3]
                    width_prior = source_priors[3]
                    width_limit = limiting_value(source_params[3], source_result[source_params[3]], width_chain, n_walkers, width_prior)

                    source_result[source_params[3]] = round_tuple((float(Angle(source_result[source_params[3]][0], units.radian).to(units.arcsec).value), \
                                                    float(Angle(source_result[source_params[3]][1], units.radian).to(units.arcsec).value)))
                    if width_limit is not None:
                        source_result[source_params[3]] = (source_result[source_params[3]][0], source_result[source_params[3]][1], \
                                                       (width_limit[0], sigfig.round(float(Angle(width_limit[1], units.radian).to(units.arcsec).value), sigfigs=3)))

                if source_type == 'g': # convert visibility theta to image theta in degrees and convert sigma and ratio into major and minor
                    theta_chain = source_chain[:, 5]
                    theta_prior = source_priors[5]
                    theta_limit = limiting_value('vis_theta', source_result['vis_theta'], theta_chain, n_walkers, theta_prior)
                    uvis_theta = ufloat(source_result['vis_theta'][0], source_result['vis_theta'][1])
                    uimg_theta = (uvis_theta * (180/np.pi) - 90)
                    del source_result['vis_theta']
                    source_result['theta'] = round_tuple((uimg_theta.n % 90, uimg_theta.s))
                    if theta_limit is not None:
                        source_result['theta'] = (source_result['theta'][0], source_result['theta'][1], \
                                                 (theta_limit[0], sigfig.round(float(theta_limit[1] * 180/np.pi), sigfigs=3)))

                    ratio_chain = source_chain[:, 4]
                    ratio_prior = source_priors[4]
                    ratio_limit = limiting_value('ratio', source_result['ratio'], ratio_chain, n_walkers, ratio_prior)
                    usigma_min = ufloat(source_result['sigma'][0], source_result['sigma'][1])
                    uratio = ufloat(source_result['ratio'][0], source_result['ratio'][1])
                    usigma_maj = usigma_min / uratio
                    del source_result['sigma']
                    del source_result['ratio']
                    source_result['sigma_maj'] = round_tuple((usigma_maj.n, usigma_maj.s))
                    source_result['sigma_min'] = round_tuple((usigma_min.n, usigma_min.s))
                    if ratio_limit is not None:
                        comment = ratio_limit[0]
                        if 'upper' in comment:
                            comment = comment.replace('upper', 'lower') # because of sigma_maj = sigma_min / ratio
                        elif 'lower' in comment:
                            comment = comment.replace('lower', 'upper') # because of sigma_maj = sigma_min / ratio
                        source_result['sigma_maj'] = (source_result['sigma_maj'][0], source_result['sigma_maj'][1], \
                                                     (comment, sigfig.round((source_result['sigma_maj'][0] * uratio / ratio_limit[1]).n, sigfigs=3)))

                del source_result['best']

                start += n_source_params

    if corner_plot:
        for j in range(len(all_results)):
            permutation_info = all_results[j]
            result = permutation_info['result']
            if result is None:
                continue
            chain = permutation_info['chain']
            start = 0
            end = 0
            for i in range(n_sources):
                source_key = f'source_{i+1}'
                source_result = result[source_key]
                source_type = source_result['type']
                n_params = SOURCE_TYPES[source_type][0]
                source_params = SOURCE_TYPES[source_type][4]
                end += n_params
                if i == n_sources-1:
                    fig = corner.corner(chain[:, start:], labels=source_params)
                else:
                    fig = corner.corner(chain[:, start:end], labels=source_params)
                fig.suptitle(f'Permutation {j+1}: {permutation_info["permutation"]}, source {i+1} of {n_sources}')
                start = end

    for permutation_info in all_results:
        del permutation_info['chain']

    return all_results


In [ ]:
file = fits.open('../data/uv_test/gaussian.uvfits')
cdelt1 = file[0].header['CDELT1']
cdelt1*3600

In [ ]:
##### POINT SOURCE #####
peak = 0.02
err = 0.001

file = fits.open('../data/uv_test/gaussian.uvfits')
data = file[1].data

cdelt1 = file[0].header['CDELT1']
cunit1 = file[0].header['CUNIT1']
naxis1 = file[0].header['NAXIS1']
bmaj = file[0].header['BMAJ']
bmin = file[0].header['BMIN']
int_peaks = [peak]
int_coords = [(3.0, 4.0)] # arcsec
ext_peaks = []
ext_coords = []

info = {}
info["CDELT1"] = cdelt1
info["CUNIT1"] = cunit1
info["NAXIS1"] = naxis1
info["BMAJ"] = bmaj
info["BMIN"] = bmin
info["int_peak_val"] = int_peaks
info["int_peak_coord"] = int_coords
info["ext_peak_val"] = ext_peaks
info["ext_peak_coord"] = ext_coords

vis_err = err * np.sqrt(len(data)) * np.sqrt(2)
weight = 1/(vis_err**2)

l0 = Angle(int_coords[0][0], units.arcsec).to(units.radian).value
m0 = Angle(int_coords[0][1], units.arcsec).to(units.radian).value

vis = []
new_row = ()
for row in data:
    model = p_model([peak, l0, m0], row['U'], row['V'], bmaj, np.pi * bmaj * bmin / (4 * np.log(2)))
    new_row = (row['Frequency'], row['U'], row['V'], (np.random.normal(scale=vis_err) + model.real) * weight, (np.random.normal(scale=vis_err) + model.imag) * weight, weight)
    vis.append(new_row)

sim_uv_fit(info, vis, sources=['any'])

In [ ]:
##### DOUBLE POINT SOURCE #####
peak1 = 0.05
peak2 = 0.1
err = 0.01

file = fits.open('../data/uv_test/gaussian.uvfits')
data = file[1].data

cdelt1 = file[0].header['CDELT1']
cunit1 = file[0].header['CUNIT1']
naxis1 = file[0].header['NAXIS1']
bmaj = file[0].header['BMAJ']
bmin = file[0].header['BMIN']
int_peaks = [peak1]
int_coords = [(1.0, 1.0)] # arcsec
ext_peaks = [peak2]
ext_coords = [(-10, -5)]

info = {}
info["CDELT1"] = cdelt1
info["CUNIT1"] = cunit1
info["NAXIS1"] = naxis1
info["BMAJ"] = bmaj
info["BMIN"] = bmin
info["int_peak_val"] = int_peaks
info["int_peak_coord"] = int_coords
info["ext_peak_val"] = ext_peaks
info["ext_peak_coord"] = ext_coords

vis_err = err * np.sqrt(len(data)) * np.sqrt(2)
weight = 1/(vis_err**2)

l01 = Angle(int_coords[0][0], units.arcsec).to(units.radian).value
m01 = Angle(int_coords[0][1], units.arcsec).to(units.radian).value

l02 = Angle(ext_coords[0][0], units.arcsec).to(units.radian).value
m02 = Angle(ext_coords[0][1], units.arcsec).to(units.radian).value

vis = []
new_row = ()
for row in data:
    model = p_model([peak1, l01, m01], row['U'], row['V'], bmaj, np.pi * bmaj * bmin / (4 * np.log(2)))
    model += p_model([peak2, l02, m02], row['U'], row['V'], bmaj, np.pi * bmaj * bmin / (4 * np.log(2)))
    new_row = (row['Frequency'], row['U'], row['V'], (np.random.normal(scale=vis_err) + model.real) * weight, (np.random.normal(scale=vis_err) + model.imag) * weight, weight)
    vis.append(new_row)

sim_uv_fit(info, vis, sources=['any', 'any'], additional_runs=4)

In [ ]:
# ##### TESTING ELLIPTICAL GAUSSIAN #####
# peak = 2
# err = 0.01

# file = fits.open('../data/uv_test/gaussian.uvfits')
# data = file[1].data

# cdelt1 = file[0].header['CDELT1']
# cunit1 = file[0].header['CUNIT1']
# naxis1 = file[0].header['NAXIS1']
# bmaj = file[0].header['BMAJ']
# bmin = file[0].header['BMIN']
# int_peaks = [peak]
# int_coords = [(0, 0)] # arcsec
# ext_peaks = "none"
# ext_coords = "none"

# info = {}
# info["CDELT1"] = cdelt1
# info["CUNIT1"] = cunit1
# info["NAXIS1"] = naxis1
# info["BMAJ"] = bmaj
# info["BMIN"] = bmin
# info["int_peak_val"] = int_peaks
# info["int_peak_coord"] = int_coords
# info["ext_peak_val"] = ext_peaks
# info["ext_peak_coord"] = ext_coords


# sd_x = 1 # arcsec
# sd_y = 3 # arcsec
# theta = 0 # radians

# weight = 1/err**2
# pixsize = 0.2  # Satisfy Nyquist criterion, in arcsec
# rad_pix = Angle(pixsize, 'arcsec').to(units.radian).value
# imsize = 40.0 # arcsec
# npix = int(np.round(imsize/pixsize))+1 #Make sure this is odd?

# coords = np.arange(npix)*pixsize-npix/2.0*pixsize #+0.000001
# xcoords2d = np.reshape(np.repeat(coords, npix), (npix,npix))
# ycoords2d = xcoords2d.T
# gaussimage = np.zeros((npix,npix))
# distimage = np.sqrt(xcoords2d**2.0+ycoords2d**2.0)
# gaussimage = np.exp(-0.5 * ((xcoords2d*np.cos(theta)-ycoords2d*np.sin(theta))**2 / (sd_y**2) + (xcoords2d*np.sin(theta)+ycoords2d*np.cos(theta))**2 / (sd_x**2)))
# gaussimage *= peak

# gaussvisfunc=np.fft.fftshift(np.fft.fft2(np.fft.fftshift(gaussimage)))
# vispixsize=1./(npix*pixsize/3600.0*np.pi/180.0)/1e3 #klambda
# viscoords=np.arange(npix)*vispixsize-vispixsize*npix/2.0
# visucoords2d=np.reshape(np.repeat(viscoords, npix), (npix,npix))
# visvcoords2d=visucoords2d.T
# uvdistimage=np.sqrt(visucoords2d**2.0+visvcoords2d**2.0)

# num_vis_points = len(gaussvisfunc)**2
# noise_real = np.array(np.random.normal(0, err, num_vis_points))
# noise_imag = np.array(np.random.normal(0, err, num_vis_points))

# visucoords_flat = visucoords2d.flatten()
# visvcoords_flat = visvcoords2d.flatten()
# gaussvisfunc_flat = np.array(gaussvisfunc.flatten())
# noisy_vis = gaussvisfunc_flat + noise_real + 1j*noise_imag

# vis = []
# for i in range(num_vis_points):
#     if np.random.random() < 0.5:
#         newrow = (0, float(visucoords_flat[i]), float(visvcoords_flat[i]), float(noisy_vis[i].real*weight), float(noisy_vis[i].imag*weight), weight)
#         vis.append(newrow)

# sim_uv_fit(info, vis, sources=['g'], additional_runs=0)

In [ ]:
# from matplotlib import pyplot as plt
# plt.imshow(gaussimage, origin='lower')
# print(np.max(gaussimage))

In [ ]:
uv_fit('../data/uv_test/3c84.uvfits', sources=['any'], priors=None, clean_output=True, corner_plot=True, additional_runs=2)

In [ ]:
fits.open('../data/uv_test/3c84.uvfits')[0].header['CDELT1']*3600

In [ ]:
from find_source import summary
summary('../data/uv_test/3c84.uvfits')

In [ ]:
# uv_fit('../data/uv_test/test.fits', sources=['any'], priors=None, clean_output=True, corner_plot=True, additional_runs=2)

In [ ]:
uv_fit('../data/uv_test/gaussian.uvfits', sources=['any'], clean_output=True, corner_plot=True, additional_runs=2)